# 准备数据

In [ ]:
#papermill research_workflow_mlp_fixed.ipynb  research_workflow_mlp_fixed_output.ipynb --progress-bar

In [ ]:
#  nohup papermill research_workflow_mlp_fixed.ipynb research_workflow_mlp_fixed_output.ipynb --progress-bar > papermill.log 2>&1 &

In [ ]:
# 过滤Alphalens的warning
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# 加载模块
import polars as pl

from vnpy.trader.constant import Interval

from vnpy.alpha import AlphaLab

In [ ]:
# 创建数据中心
lab: AlphaLab = AlphaLab("./lab/crypto_1m")


In [ ]:
# 设置任务参数
name = "300_mlp_crypto_1m"
index_symbol: str = "CRYPTO_INDEX_1M"
start: str = "2020-01-01"
end: str = "2025-09-03"
interval: Interval = Interval.MINUTE
extended_days: int = 100

In [ ]:
# 加载所有成分股代码
component_symbols: list[str] = lab.load_component_symbols(index_symbol, start, end)[:10]

In [ ]:
component_symbols=[ i for i in component_symbols if not i.startswith("FUSDT")]

In [ ]:
print(component_symbols)


# 特征计算

In [ ]:
# 加载模块
from datetime import datetime
from functools import partial

from vnpy.trader.constant import Interval

from vnpy.alpha.dataset import (
    AlphaDataset,
    process_drop_na,
    process_robust_zscore_norm,
    process_fill_na,
    process_cs_rank_norm,
    to_datetime
)
from vnpy.alpha.dataset.datasets.alpha_158 import Alpha158

In [ ]:
# 加载成分股数据
component_symbols=component_symbols  
df: pl.DataFrame = lab.load_bar_df(component_symbols, interval, start, end, extended_days)

In [ ]:
# 设置数据时间段
train_period: tuple[str, str] = ("2020-01-01", "2023-12-31")
valid_period: tuple[str, str] = ("2024-01-01", "2024-12-31")
test_period: tuple[str, str] = ("2025-01-01", "2025-09-03")

In [ ]:
df = df.unique(subset=["datetime", "vt_symbol"], keep="first")
print(df.head())

In [ ]:
# 创建数据集对象
dataset: AlphaDataset = Alpha158(
    df,
    train_period=train_period,
    valid_period=valid_period,
    test_period=test_period,
)

In [ ]:
# 添加数据预处理器
fit_start_time: datetime = to_datetime(train_period[0])
fit_end_time: datetime = to_datetime(train_period[1])
print(f"fit_start_time: {fit_start_time}")
print(f"fit_end_time: {fit_end_time}")
dataset.add_processor("infer", partial(process_robust_zscore_norm, fit_start_time=fit_start_time, fit_end_time=fit_end_time))
dataset.add_processor("infer", partial(process_fill_na, fill_value=0, fill_label=False))

dataset.add_processor("learn", partial(process_drop_na, names=["label"]))
dataset.add_processor("learn", partial(process_cs_rank_norm, names=["label"]))

In [ ]:
# 收集指数成分过滤器
filters: dict[str, list[str]] = lab.load_component_filters(index_symbol, start, end)

In [ ]:
# 准备特征和标签数据
dataset.prepare_data(max_workers=1)

In [ ]:
# 特征表现分
#dataset.show_feature_performance("kmid")


In [ ]:
lab.save_dataset(name, dataset)

# 模型训练

In [ ]:
# 加载模块
import numpy as np

from vnpy.alpha import Segment, AlphaDataset, AlphaModel

from vnpy.alpha.model.models.mlp_model import MlpModel

In [ ]:
dataset: AlphaDataset = lab.load_dataset(name)

In [ ]:
# 创建模型对象
# 注意：需要根据实际特征数量调整input_size
# feature_count = len(dataset.expressions)  # 使用实际特征数量
# kwargs = {
#     "input_size": feature_count,  # 根据实际特征数量调整
#     "hidden_sizes": (64,),  # 减小网络规模
#     "lr": 0.002,
#     "optimizer": "adam",
#     "n_epochs": 100,  # 减少训练轮数，加快测试
#     "batch_size": 1024,  # 减小批量大小
#     "weight_decay": 0.0002,
#     "seed": 42
# }
kwargs = {
    "input_size": 158,
    "hidden_sizes": (256,),
    "lr": 0.002,
    "optimizer": "adam",
    "n_epochs": 8000,
    "batch_size": 8192,
    "weight_decay": 0.0002,
    "seed": 42
}
model: AlphaModel = MlpModel(**kwargs)

In [ ]:
# 查看模型细节
model.fit(dataset)
model.detail()
lab.save_model(name, model)

# 预测信号

In [ ]:
model: AlphaModel = lab.load_model(name)

In [ ]:
# 用模型在测试集上预测
pre: np.ndarray = model.predict(dataset, Segment.TEST)

# 加载测试集数据
df_t: pl.DataFrame = dataset.fetch_feat(Segment.TEST)

# 合并预测信号列
df_t = df_t.with_columns(pl.Series(pre).alias("signal"))

# 提取信号数据
signal: pl.DataFrame = df_t["datetime", "vt_symbol", "signal"]

In [ ]:
dataset.show_signal_performance(signal)

In [ ]:
# 保存信号数据
lab.save_signal(name, signal)

# 策略回测

In [ ]:
# 加载模块
import importlib
from datetime import datetime

from vnpy.alpha.strategy import BacktestingEngine

import vnpy.alpha.strategy.strategies.equity_demo_strategy as equity_demo_strategy

In [ ]:
# 重载策略类
importlib.reload(equity_demo_strategy)
EquityDemoStrategy = equity_demo_strategy.EquityDemoStrategy

In [ ]:
# 从文件加载信号数据
signal = lab.load_signal(name)

In [ ]:
engine = BacktestingEngine(lab)

# 设置回测参数
engine.set_parameters(
    vt_symbols=component_symbols,  # 使用减少后的符号列表
    interval=Interval.MINUTE,
    start=datetime(2025, 1, 1),
    end=datetime(2025, 10, 31),
    capital=100000000
)

# 添加策略实例
setting = {"top_k": 3, "n_drop": 2, "hold_thresh": 3}  # 调整参数以适应较少的符号
engine.add_strategy(EquityDemoStrategy, setting, signal)

In [ ]:
# 执行回测任务

engine.load_data()
engine.run_backtesting()
engine.calculate_result()
engine.calculate_statistics()
engine.show_chart()

In [ ]:
engine.show_performance(benchmark_symbol=index_symbol)